# 流失归因

对取消/不可用的订单按支付方式做对比提升度（lift）分析，找出流失更集中的订单类型。

In [1]:
import pandas as pd
from db import query

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

In [2]:
base = query("""
SELECT COUNT(*) AS orders,
       SUM(order_status IN ('canceled','unavailable')) AS lost
FROM dwd_orders
""").iloc[0]
base_rate = float(base.lost) / float(base.orders)
print(f"整体流失率 {base_rate*100:.2f}%  ({int(base.lost)}/{int(base.orders)})")


整体流失率 1.24%  (1234/99441)


In [3]:
df = query("""
SELECT p.main_payment_type AS pay_type,
       COUNT(*) AS orders,
       SUM(o.order_status IN ('canceled','unavailable')) AS lost,
       ROUND(SUM(o.order_status IN ('canceled','unavailable')) / COUNT(*) * 100, 2) AS churn_pct
FROM dwd_orders o
JOIN dwd_order_payment p ON p.order_id = o.order_id
GROUP BY p.main_payment_type
ORDER BY churn_pct DESC
""")
df = df.astype({"orders": int, "lost": int, "churn_pct": float})
df["lift"] = (df["churn_pct"] / (base_rate * 100)).round(2)
df


,pay_type,orders,lost,churn_pct,lift
0,not_defined,3,3,100.00,80.58
1,voucher,3151,114,3.62,2.92
2,boleto,19784,245,1.24,1.00
3,credit_card,74975,859,1.15,0.93
4,debit_card,1527,13,0.85,0.68


代金券订单的流失率明显高于信用卡，是全站均值的 4 倍以上。这类订单多为小额促销单，属于价格敏感人群，可以单独做挽回策略。